In [ ]:
import sys
sys.path.append("../")
import numpy as np

#load the necessary odometry modules
from odometry.datasets.map_handler import MapHandler
from odometry.datasets.radnav_ds import radnavDS
from odometry.test_benches.icp2D_localization_kalman_tb import icp2DLocalizationKalmanTB
from odometry.localization.icp2D_localization import icp2DLocalization
from odometry.plotting.plotter_kalman import PlotterKalman

In [ ]:
from dotenv import load_dotenv
import os

#loading enviroment variables
load_dotenv()
DATASET_PATH=os.getenv("DATASET_DIRECTORY")
MAP_DIRECTORY=os.getenv("MAP_DIRECTORY")

#setup the datasets
dataset = radnavDS(
    dataset_path=DATASET_PATH + "/athena_test_2/",
    radar_folder="radar_combined",
    lidar_folder="lidar",
    camera_folder="camera",
    imu_orientation_folder="imu_data",
    imu_full_folder="imu_data_full",
    vehicle_vel_folder="vehicle_vel"
)

#load the map
map_handler = MapHandler(
    maps_folder=MAP_DIRECTORY,
    map_file="athena.yaml"
)

In [ ]:
#initialize the localizers
radar_odometry = icp2DLocalization(
    icp_matching_distance_threshold=0.3,#originally 0.5
    icp_best_points_percentile=40,
    icp_convergence_translation_threshold=1e-3,
    icp_convergence_rotation_threshold=1e-4,
    icp_point_pairs_threshold=7, #originally 5
    icp_max_iterations=20,
    self_detection_radius_m=0.25
)

lidar_odometry = icp2DLocalization(
    icp_matching_distance_threshold=0.6,
    icp_best_points_percentile=50,
    icp_convergence_translation_threshold=1e-3,
    icp_convergence_rotation_threshold=1e-4,
    icp_point_pairs_threshold=10,
    icp_max_iterations=20,
    self_detection_radius_m=0.25
)

In [ ]:
#initialize the test bench
test_bench = icp2DLocalizationKalmanTB(
    localizer=radar_odometry,
    gt_localizer=lidar_odometry,
    map_handler=map_handler,
    dataset=dataset
)

start_heading = np.deg2rad(0)
start_pose = np.array([0.0,0.0])

new_heading_rad,new_pose_m = test_bench.init_localization(
    est_start_heading_rad=np.deg2rad(0),
    est_start_pose_m=np.array([0.0,0.0]),
    show=True
)

In [ ]:
#initialize the kalman filter
start_time_s = test_bench.get_dataset_start_time(idx=0)
print("dataset start time: {}s".format(start_time_s))

test_bench.init_filter(
    est_start_heading_rad=new_heading_rad,
    est_start_position_m=new_pose_m,
    start_time_s=start_time_s
)

In [ ]:
lidar_cloud = dataset.get_lidar_point_cloud(idx=0)
radar_cloud = dataset.get_radar_detections(idx=0)

print("radar: {}, lidar:{}".format(radar_cloud.shape[0],lidar_cloud.shape[0]))

In [ ]:
test_bench.run()

In [ ]:
test_bench.plotter.plot_position_history_m(
    history_position_m=test_bench.history_position_m,
    history_position_m_gt=test_bench.history_position_m_gt,
    idx=0,
    ax=None,
    show=True
)

In [ ]:
test_bench.plotter.plot_heading_history_deg(
    history_heading_deg=test_bench.history_heading_deg,
    history_heading_deg_gt=test_bench.history_heading_deg_gt,
    idx =0,
    ax=None,
    show=True
)

In [ ]:
test_bench.analyze()

In [ ]:
#plot kalman state history
kalman_plotter = PlotterKalman()

kalman_plotter.plot_kalman_result_history(
    x_history=np.array(test_bench.history_filter_est),
    p_history=np.array(test_bench.history_filter_p),
    idx=0
)

In [ ]:
kalman_plotter.plot_chi_2_resp(
    g_thresh=test_bench.filter.g_thresh[2],
    g_hist=np.array(test_bench.history_filter_g),
    idx=0
)

In [ ]:
kalman_plotter.plot_residual_hist(
    y_hist=test_bench.history_filter_y,
    idx=0
)

In [ ]:
#look at the vehicle velocity data
vel_data = dataset.get_vehicle_vel_data(idx=0)

for i in range(vel_data.shape[0]):

    out_str = "sample: {}, time: {:.2f}s, lin_vel: {:.2f}m/s, ang_vel: {:.2f}rad/s".format(
        i,
        vel_data[i,0],
        vel_data[i,1],
        vel_data[i,2]
    )

    print(out_str)

In [ ]:
# look at the imu raw data
imu_data_full = dataset.get_imu_full_data(idx=0)

for i in range(imu_data_full.shape[0]):

    out_str = "idx: {}, time: {:.2f}s, gyro:[{:.2f},{:.2f},{:.2f}] rad/s, acc: [{:.2f},{:.2f},{:.2f}]".format(
        i,
        imu_data_full[i,0],
        imu_data_full[i,1],
        imu_data_full[i,2],
        imu_data_full[i,3],
        imu_data_full[i,4],
        imu_data_full[i,5],
        imu_data_full[i,6]
    )

    print(out_str)